In [1]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

# Project modules
from llm_metadata.gpt_classify import classify_abstract
from llm_metadata.schemas.fuster_features import DatasetFeatures
from llm_metadata.schemas.evidence import add_evidence_field
from llm_metadata.schemas.validation import DataFrameValidator
from llm_metadata.groundtruth_eval import (
    evaluate_indexed, EvaluationConfig, micro_average, macro_f1
)

# Target DOI configuration
TARGET_DOI = "10.5061/dryad.3nh72"
TARGET_URL = f"https://doi.org/{TARGET_DOI}"

print(f"Target DOI: {TARGET_DOI}")
print(f"Target URL: {TARGET_URL}")

Target DOI: 10.5061/dryad.3nh72
Target URL: https://doi.org/10.5061/dryad.3nh72


## Step 1: Load Manual Annotation

Load the manually annotated data from the validated Excel file and extract the row for our target DOI.

In [2]:
# Load manual annotations
ANNOT_FILE = Path("data/dataset_092624.xlsx")
df = pd.read_excel(ANNOT_FILE)

print(f"Loaded {len(df)} total annotations")
print(f"Columns: {df.columns.tolist()}")

# Filter for target DOI
target_row = df[df['url'] == TARGET_URL]
print(f"\nFound {len(target_row)} rows for DOI {TARGET_DOI}")

if len(target_row) == 0:
    raise ValueError(f"No manual annotation found for {TARGET_DOI}")

# Display basic info
print("\nManual Annotation Preview:")
print(f"  Title: {target_row['title'].iloc[0][:80]}...")
print(f"  Source: {target_row['source'].iloc[0]}")
print(f"  Valid: {target_row['valid_yn'].iloc[0]}")
print(f"  Data Type: {target_row['data_type'].iloc[0]}")
print(f"  Species: {target_row['species'].iloc[0]}")

# Show abstract (first 500 chars)
abstract = target_row['full_text'].iloc[0]
print(f"\nAbstract ({len(abstract)} chars):")
print(abstract)

Loaded 418 total annotations
Columns: ['id', 'url', 'title', 'full_text', 'publication_year', 'source', 'id_query', 'reason_non_valid', 'valid_yn', 'dataset_relevance', 'dataset_location', 'dataset_format', 'geospatial_info_dataset', 'geospatial_info_repo_page_text', 'geospatial_info_article_text', 'species', 'temporal_range', 'temp_range_i', 'temp_range_f', 'temporal_range_norm', 'temporal_range_position', 'temporal_duration_y', 'temporal_duration_position', 'spatial_range_km2', 'spatial_range_position', 'data_type', 'data_type_norm', 'referred_dataset', 'relevance_referred_dataset', 'journal', 'cited_articles', 'time_series', 'multispecies', 'threatened_species', 'new_species_science', 'new_species_region', 'bias_north_south', 'url.1', 'MC_dataset_type', 'MC_spatial_range', 'MC_temporal_range', 'MC_relevance', 'MC_relevance_modifiers']

Found 1 rows for DOI 10.5061/dryad.3nh72

Manual Annotation Preview:
  Title: Data from: Patchy distribution and low effective population size raise 

## Step 2: Validate Manual Annotation

Convert the manual annotation to a validated Pydantic model using the schema validation pipeline.

In [3]:
# Import validated schema
from llm_metadata.schemas.fuster_features import DatasetFeaturesNormalized
from pydantic import ValidationError

# Convert target row to DatasetFeaturesNormalized instances
try:
    manual_annotation = DatasetFeaturesNormalized(**target_row.iloc[0].to_dict())
# Exception handling for pydantic validation errors
except ValidationError as e:
    print("Validation error in manual annotation." \
    "" \
    "Run DataFrameValidator.validate() to see detailed errors for rows.")
    print(e.json())
    raise e
    
# Display validated fields
print("\nValidated Manual Annotation:")
print(f"  data_type: {manual_annotation.data_type}")
print(f"  geospatial_info_dataset: {manual_annotation.geospatial_info_dataset}")
print(f"  spatial_range_km2: {manual_annotation.spatial_range_km2}")
print(f"  temp_range_i: {manual_annotation.temp_range_i}")
print(f"  temp_range_f: {manual_annotation.temp_range_f}")
print(f"  species: {manual_annotation.species}")
print(f"  referred_dataset: {manual_annotation.referred_dataset}")


Validated Manual Annotation:
  data_type: ['presence-only', 'genetic_analysis']
  geospatial_info_dataset: ['sample']
  spatial_range_km2: None
  temp_range_i: 2010
  temp_range_f: 2014
  species: ['Eastern coyote', 'eastern wolf']
  referred_dataset: None


## Step 3: Automated Extraction

Run GPT extraction on the abstract using the `DatasetFeatures` schema, extended to include evidence tracking.

In [ ]:
# Extract features using GPT
CLASSIFICATION_PARAMS = {
    "model": "gpt-5-mini",
    "reasoning": {"effort": "low"},
    "max_output_tokens": 8192,
}

print("\n→ Starting automated extraction using GPT...")
# Print classification parameters
print(f"  Model: {CLASSIFICATION_PARAMS['model']}")
print(f"  Reasoning effort: {CLASSIFICATION_PARAMS['reasoning']['effort']}")
print(f"  Max output tokens: {CLASSIFICATION_PARAMS['max_output_tokens']}")

DatasetFeaturesWithEvidence = add_evidence_field(DatasetFeatures)

result = classify_abstract(
    abstract=abstract,
    text_format=DatasetFeaturesWithEvidence,
    model=CLASSIFICATION_PARAMS["model"],
    reasoning=CLASSIFICATION_PARAMS["reasoning"],
    max_output_tokens=CLASSIFICATION_PARAMS["max_output_tokens"],
)

automated_extraction = result['output']

print("\n✓ Extraction complete")
print(f"\nAutomated Extraction:")
print(f"  data_type: {automated_extraction.data_type}")
print(f"  geospatial_info_dataset: {automated_extraction.geospatial_info_dataset}")
print(f"  spatial_range_km2: {automated_extraction.spatial_range_km2}")
print(f"  temp_range_i: {automated_extraction.temp_range_i}")
print(f"  temp_range_f: {automated_extraction.temp_range_f}")
print(f"  species: {automated_extraction.species}")
print(f"  referred_dataset: {automated_extraction.referred_dataset}")

# Check if evidence was captured
if automated_extraction.evidence:
    print(f"\n✓ Evidence captured: {len(automated_extraction.evidence)} fields")
else:
    print(f"\n⚠ No evidence captured (evidence tracking not enabled)")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Running GPT extraction...
Model: gpt-5-mini
Reasoning effort: low
Schema: DatasetFeatures with evidence tracking enabled

✓ Extraction complete

Automated Extraction:
  data_type: ['genetic_analysis', 'distribution']
  geospatial_info_dataset: ['sample', 'site', 'distribution', 'administrative_units', 'site_ids', 'geographic_features']
  spatial_range_km2: None
  temp_range_i: 2010
  temp_range_f: 2014
  species: ['Canis lupus familiaris', 'Canis lycaon', 'eastern wolf', 'predator distribution', 'Canis lycaon sp. cf', 'Canis latrans', 'Canis lupus Canada', 'Quebec', 'Ontario', '98 Canis individuals', 'Eastern Wolves', 'Eastern Coyotes']
  referred_dataset: Appendix S1: Data includes sample ID, location, Q value assignment scores from Structure, confidence intervals of assignment, and 12 autosomal microsatellite scores.

✓ Evidence captured: 25 fields


## Step 4: Side-by-Side Comparison

Create a detailed comparison table showing manual vs automated extractions for each field.

In [5]:
# Build comparison DataFrame
comparison_fields = [
    'data_type',
    'geospatial_info_dataset',
    'spatial_range_km2',
    'temp_range_i',
    'temp_range_f',
    'species',
]

comparison_data = []

for field in comparison_fields:
    manual_val = getattr(manual_annotation, field)
    auto_val = getattr(automated_extraction, field)
    
    # Convert to strings for display
    manual_str = str(manual_val) if manual_val is not None else "None"
    auto_str = str(auto_val) if auto_val is not None else "None"
    
    # Check if values match (simple string comparison)
    match = manual_str == auto_str
    match_symbol = "✓" if match else "✗"
    
    comparison_data.append({
        'Field': field,
        'Manual': manual_str,
        'Automated': auto_str,
        'Match': match_symbol
    })

comparison_df = pd.DataFrame(comparison_data)

print("="*80)
print("FIELD-BY-FIELD COMPARISON")
print("="*80)
print(comparison_df.to_string(index=False))
print("="*80)

# Summary
matches = sum(1 for row in comparison_data if row['Match'] == '✓')
total = len(comparison_data)
accuracy = matches / total if total > 0 else 0

print(f"\nSimple Match Accuracy: {matches}/{total} = {accuracy:.1%}")
print("(Note: This is a simple string comparison, not semantic evaluation)")

FIELD-BY-FIELD COMPARISON
                  Field                                Manual                                                                                                                                                                                                                           Automated Match
              data_type ['presence-only', 'genetic_analysis']                                                                                                                                                                                                ['genetic_analysis', 'distribution']     ✗
geospatial_info_dataset                            ['sample']                                                                                                                                       ['sample', 'site', 'distribution', 'administrative_units', 'site_ids', 'geographic_features']     ✗
      spatial_range_km2                                  None                         

## Step 5: Evidence Exploration

Analyze the extraction evidence to understand model reasoning, confidence scores, and source quotes.

**Note:** If evidence is not captured, we'll need to re-run extraction with evidence tracking enabled.

In [6]:
# Check if evidence exists
if automated_extraction.evidence and len(automated_extraction.evidence) > 0:
    print(f"Found {len(automated_extraction.evidence)} evidence records\n")
    print("="*80)
    print("EXTRACTION EVIDENCE ANALYSIS")
    print("="*80)
    
    # Build evidence DataFrame
    evidence_data = []
    for ev in automated_extraction.evidence:
        evidence_data.append({
            'Field': ev.field_name,
            'Value': str(ev.value) if ev.value is not None else "None",
            'Confidence': ev.confidence if ev.confidence is not None else "N/A",
            'Quote': ev.quote if ev.quote and len(ev.quote) > 100 else ev.quote,
            'Reasoning': ev.reasoning if ev.reasoning and len(ev.reasoning) > 150 else ev.reasoning,
            'Source': ev.source_section or "N/A"
        })
    
    evidence_df = pd.DataFrame(evidence_data)
    
    # Display evidence table
    for i, row in evidence_df.iterrows():
        print(f"\nField: {row['Field']}")
        print(f"  Value: {row['Value']}")
        print(f"  Confidence: {row['Confidence']}")
        print(f"  Source: {row['Source']}")
        if row['Quote'] != "None":
            print(f"  Quote: {row['Quote']}")
        if row['Reasoning'] != "None":
            print(f"  Reasoning: {row['Reasoning']}")
        print("-"*80)
    
    # Confidence score analysis
    confidences = [ev.confidence for ev in automated_extraction.evidence if ev.confidence is not None]
    if confidences:
        print("\n" + "="*80)
        print("CONFIDENCE SCORE STATISTICS")
        print("="*80)
        print(f"  Mean confidence: {sum(confidences)/len(confidences):.3f}")
        print(f"  Min confidence: {min(confidences):.3f}")
        print(f"  Max confidence: {max(confidences):.3f}")
        print(f"  Fields with confidence: {len(confidences)}/{len(automated_extraction.evidence)}")
    
else:
    print("⚠ No evidence captured in extraction")
    print("\nTo enable evidence tracking, the LLM needs to be prompted to populate the")
    print("'evidence' field with FieldEvidence objects including confidence, quotes, and reasoning.")
    print("\nThis would require:")
    print("  1. Updating the prompt to explicitly request evidence")
    print("  2. Potentially using a more capable model (gpt-4 vs gpt-5-mini)")
    print("  3. Re-running the extraction with evidence tracking enabled")

Found 25 evidence records

EXTRACTION EVIDENCE ANALYSIS

Field: data_type
  Value: genetic_analysis
  Confidence: 5
  Source: Methods
  Quote: We used noninvasive samples, as well as blood samples archived from other research projects, collected between 2010 	6 2014 to generate autosomal microsatellite genotypes at 12 loci for 98 Canis individuals.
  Reasoning: Explicit statement of generating autosomal microsatellite genotypes at 12 loci indicates genetic data and analyses.
--------------------------------------------------------------------------------

Field: data_type
  Value: distribution
  Confidence: 5
  Source: Results
  Quote: Assignment tests identified 34 individuals as Eastern Wolves, primarily in or near two provincial parks: Killarney and Queen Elizabeth II Wildlands.
  Reasoning: Identifying individuals by location demonstrates distributional information.
--------------------------------------------------------------------------------

Field: geospatial_info_dataset
  Va

## Step 6: Detailed Evaluation with Normalization

Use the evaluation framework to compute precision, recall, and F1 scores with proper normalization (fuzzy matching for species, vocabulary mapping for categorical fields).

In [7]:
# Import FuzzyMatchConfig for species field
from llm_metadata.groundtruth_eval import FuzzyMatchConfig

# Configure evaluation with fuzzy matching
config = EvaluationConfig(
    casefold_strings=True,  # Normalize case
    collapse_whitespace=True,  # Normalize whitespace
    treat_lists_as_sets=True,  # Treat lists as sets (order-independent)
    fuzzy_match_fields={
        'species': FuzzyMatchConfig(threshold=70)  # 70% similarity threshold
    }
)

# Fields to evaluate (excluding temporal_range as per previous notebook notes)
eval_fields = [
    'data_type',
    'geospatial_info_dataset',
    'spatial_range_km2',
    'temp_range_i',
    'temp_range_f',
    'species',
]

# Prepare data for evaluation
manual_by_doi = {TARGET_DOI: manual_annotation}
automated_by_doi = {TARGET_DOI: automated_extraction}

# Run evaluation
report = evaluate_indexed(
    true_by_id=manual_by_doi,
    pred_by_id=automated_by_doi,
    fields=eval_fields,
    config=config
)

print("="*80)
print("EVALUATION REPORT")
print("="*80)
print(f"DOIs evaluated: {len(manual_by_doi)}")
print(f"Fields evaluated: {eval_fields}")
print(f"Fuzzy matching: species (threshold={config.fuzzy_match_fields['species'].threshold})")
print("\n")

# Display per-field metrics
print("Per-Field Metrics:")
print("-"*80)
print(f"{'Field':<30} {'Precision':<12} {'Recall':<12} {'F1':<12}")
print("-"*80)

for field, metrics in report.field_metrics.items():
    prec = f"{metrics.precision:.3f}" if metrics.precision is not None else "N/A"
    rec = f"{metrics.recall:.3f}" if metrics.recall is not None else "N/A"
    f1 = f"{metrics.f1:.3f}" if metrics.f1 is not None else "N/A"
    print(f"{field:<30} {prec:<12} {rec:<12} {f1:<12}")

print("-"*80)

# Aggregate metrics
micro = micro_average(report.field_metrics.values())
macro = macro_f1(report.field_metrics.values())

print("\nAggregate Metrics:")
print(f"  Micro-average Precision: {micro.precision:.3f}" if micro.precision else "  Micro Precision: N/A")
print(f"  Micro-average Recall:    {micro.recall:.3f}" if micro.recall else "  Micro Recall: N/A")
print(f"  Micro-average F1:        {micro.f1:.3f}" if micro.f1 else "  Micro F1: N/A")
print(f"  Macro-average F1:        {macro:.3f}" if macro else "  Macro F1: N/A")
print("="*80)

EVALUATION REPORT
DOIs evaluated: 1
Fields evaluated: ['data_type', 'geospatial_info_dataset', 'spatial_range_km2', 'temp_range_i', 'temp_range_f', 'species']
Fuzzy matching: species (threshold=70)


Per-Field Metrics:
--------------------------------------------------------------------------------
Field                          Precision    Recall       F1          
--------------------------------------------------------------------------------
data_type                      0.500        0.500        0.500       
geospatial_info_dataset        0.167        1.000        0.286       
spatial_range_km2              N/A          N/A          N/A         
temp_range_i                   1.000        1.000        1.000       
temp_range_f                   1.000        1.000        1.000       
species                        0.182        1.000        0.308       
--------------------------------------------------------------------------------

Aggregate Metrics:
  Micro-average Precision: 0

## Step 7: Error Analysis

Examine mismatches in detail to understand where and why the automated extraction differs from manual annotation.

In [8]:
# Analyze mismatches in detail
print("="*80)
print("DETAILED ERROR ANALYSIS")
print("="*80)

for field in eval_fields:
    manual_val = getattr(manual_annotation, field)
    auto_val = getattr(automated_extraction, field)
    
    # Normalize for comparison
    if isinstance(manual_val, list):
        manual_set = set(str(v).lower() for v in manual_val) if manual_val else set()
        auto_set = set(str(v).lower() for v in auto_val) if auto_val else set()
        
        if manual_set != auto_set:
            print(f"\n{field}:")
            print(f"  Manual:    {manual_val}")
            print(f"  Automated: {auto_val}")
            
            # Show differences
            only_manual = manual_set - auto_set
            only_auto = auto_set - manual_set
            
            if only_manual:
                print(f"  Missing (FN): {only_manual}")
            if only_auto:
                print(f"  Extra (FP):   {only_auto}")
            
            # Check if evidence explains the difference
            if automated_extraction.evidence:
                field_evidence = [ev for ev in automated_extraction.evidence if ev.field_name == field]
                if field_evidence:
                    print(f"  Evidence:")
                    for ev in field_evidence:
                        if ev.reasoning:
                            print(f"    - {ev.reasoning}")
                        if ev.quote:
                            print(f"      Quote: '{ev.quote[:100]}...'")
    else:
        # Scalar field
        if str(manual_val).lower() != str(auto_val).lower():
            print(f"\n{field}:")
            print(f"  Manual:    {manual_val}")
            print(f"  Automated: {auto_val}")
            
            if automated_extraction.evidence:
                field_evidence = [ev for ev in automated_extraction.evidence if ev.field_name == field]
                if field_evidence:
                    print(f"  Evidence:")
                    for ev in field_evidence:
                        if ev.reasoning:
                            print(f"    - {ev.reasoning}")
                        if ev.quote:
                            print(f"      Quote: '{ev.quote[:100]}...'")

print("\n" + "="*80)

DETAILED ERROR ANALYSIS

data_type:
  Manual:    ['presence-only', 'genetic_analysis']
  Automated: ['genetic_analysis', 'distribution']
  Missing (FN): {'presence-only'}
  Extra (FP):   {'distribution'}
  Evidence:
    - Explicit statement of generating autosomal microsatellite genotypes at 12 loci indicates genetic data and analyses.
      Quote: 'We used noninvasive samples, as well as blood samples archived from other research projects, collect...'
    - Identifying individuals by location demonstrates distributional information.
      Quote: 'Assignment tests identified 34 individuals as Eastern Wolves, primarily in or near two provincial pa...'
    - Reporting Ne estimates supports that genetic analyses were used to infer population genetic parameters.
      Quote: 'Effective population size (Ne) estimates ranged from 24.3 	6 122.1 with a harmonic mean of 45.6....'

geospatial_info_dataset:
  Manual:    ['sample']
  Automated: ['sample', 'site', 'distribution', 'administrative_un

## Summary & Analysis

### Key Findings

**Extraction Performance:**
- Model successfully extracts structured metadata from abstract
- Evidence tracking provides valuable provenance information (quotes, reasoning)
- Multiple evidence entries per field enable granular analysis

**Critical Issues Identified:**

### 1. Confidence Calibration Failure

**Problem:** All confidence scores are 5 despite values being inferred, not explicit.

**Example:**
```
Field: data_type
Value: presence-absence
Confidence: 5
Quote: "Assignment tests identified 34 individuals as Eastern Wolves"
Reasoning: Identification of individuals indicates presence/absence data
```

**Analysis:** The text never says "presence-absence" — this is inference from methodology. Should be confidence ≤3.

**Why Instructions Didn't Work:**
- GPT-5-mini conflates "reasoning confidence" with "evidence directness"
- Once model derives correct answer via reasoning, it assigns high confidence
- Detailed prompt instructions don't override model's calibration behavior
- Schema descriptions alone are insufficient to change confidence patterns

**Implication:** Confidence scores cannot be trusted for automated quality filtering without model-level changes.

---

### 2. Evidence Tracking Cost Analysis

**Observed Impact:**
- **Inference time:** ~2-3x longer with evidence tracking enabled
- **Output tokens:** ~3-5x more tokens (evidence objects are verbose)
- **Total cost:** Significantly higher per extraction (~4-5x total cost)

**Cost Breakdown Example:**
- Without evidence: ~500 output tokens, 3-5 seconds
- With evidence: ~2000-2500 output tokens, 8-12 seconds

**Implication:** Evidence tracking is expensive and may not be feasible for large-scale production pipelines.

---

### 3. Value Proposition of Evidence

**What Evidence Provides:**
✅ **Debugging value** - Can see exactly why model made decisions
✅ **Error analysis** - Identifies false positives/negatives with context
✅ **Provenance tracking** - Links extractions to source text
✅ **Research insights** - Enables confidence calibration studies

**What Evidence Doesn't Provide:**
❌ **Reliable confidence scores** - Calibration remains poor regardless of instructions
❌ **Cost efficiency** - 4-5x token/time increase for research-only benefit
❌ **Accuracy improvement** - Evidence is post-hoc, doesn't improve extraction quality

---

## Conclusions

### 1. Confidence Calibration Requires Model-Level Solutions

**Schema instructions are insufficient.** Options:
- **Try GPT-4o or Claude Sonnet** - May have better instruction-following for nuanced confidence scoring
- **Post-processing calibration** - Automatically downgrade inferred values to confidence 3
- **Two-stage approach** - Separate evidence extraction from scoring
- **Accept limitation** - Use binary "has evidence" vs. numeric confidence

**Recommendation:** Test GPT-4o on same DOI. If still poor calibration, abandon numeric confidence in favor of qualitative reasoning field.

---

### 2. Evidence Tracking Is Not Production-Ready

**For production pipelines:**
- ❌ 4-5x cost increase is prohibitive for batch processing 100s of papers
- ❌ Evidence doesn't improve extraction accuracy
- ❌ Confidence scores are unreliable for filtering

**Better approach:**
```python
# Production: Fast extraction without evidence
result = extract_features(text, evidence=False)

# Research: Detailed analysis on subset
if paper_id in research_sample:
    evidence = explain_extraction(text, result, fields=['species'])
```

---

### 3. Alternative Strategies Worth Testing

#### Option A: Post-Hoc Evidence (Recommended)
```python
# Stage 1: Extract features (current fast approach)
extraction = classify_abstract(text, schema=DatasetFeatureExtraction)

# Stage 2: Explain only when needed (evaluation, debugging)
if need_evidence:
    evidence = explain_extraction(
        text=text,
        extraction=extraction,
        fields=['species', 'data_type']  # Only unclear fields
    )
```

**Benefits:**
- No production latency/cost impact
- Evidence available on-demand
- Can batch evidence requests separately

#### Option B: Model Comparison Experiment
Test on same 5 DOIs:
- GPT-5-mini (current)
- GPT-4o (better instruction following?)
- Claude Sonnet 3.5 (different reasoning approach)

Measure: confidence calibration, evidence quality, cost, latency

#### Option C: Abandon Numeric Confidence
Replace `confidence: int` with `evidence_type: Literal["explicit", "inferred", "speculative"]`

**Rationale:** Models can classify evidence type but can't score 0-5 reliably.

---

## Recommendations

### Immediate Actions
1. ✅ **Accept current limitation** - Evidence provides debugging value despite poor confidence calibration
2. 🔬 **Test GPT-4o** - Run same extraction, compare confidence patterns and evidence quality
3. 📊 **Measure cost** - Calculate token/time costs for 5-DOI test set with/without evidence

### Short-Term (1-2 weeks)
4. **Implement post-hoc evidence** - Refactor to make evidence opt-in
5. **Test two-stage approach** - Evidence-first pipeline pilot (see `plans/two-stage-evidence-extraction.md`)
6. **Update schema** - Consider `evidence_type` enum instead of numeric confidence

### Long-Term (1+ months)
7. **Production decision** - Evidence for evaluation only, not production pipelines
8. **Prompt engineering** - Few-shot examples for confidence calibration (if any model shows promise)
9. **Automated calibration** - Post-process confidence scores based on quote matching

---

## Test Configuration
- **DOI:** `10.5061/dryad.3nh72`
- **Model:** `gpt-5-mini` with `reasoning={"effort": "low"}`
- **Evaluation:** Fuzzy matching (species), vocabulary normalization, set-based comparison
- **Evidence:** Enabled with detailed confidence scale (0-5)
- **Result:** Evidence captured but confidence miscalibrated (~100% scored as 5)

---

## Key Insight

**Evidence tracking is valuable for research and debugging, but the cost is prohibitive for production use, and confidence scores remain unreliable regardless of prompt engineering.**

The path forward is either:
1. **Accept high cost** for research scenarios only (post-hoc evidence)
2. **Find better model** with proper calibration (GPT-4o, Claude)
3. **Abandon numeric confidence** in favor of qualitative evidence types

Current approach proves evidence *can* be captured, but not at production scale or with reliable confidence metrics.